# H1：参数化场景批量生成 → esmini 批量跑
# H1: Parameterized Batch Scenario Generation & Batch esmini Runs

## 一、实验原理 | Principle

### 1.1 参数化场景：从"一个场景"到"场景空间"
A1 只做了一个 cut-in 场景（固定初始距离、固定目标车速），结论只对那一个点成立。
真实的测试需要覆盖**场景空间**：把关键参数连续/离散化，批量生成、批量跑、统计分析。

本实验采用**全因子实验设计**（Full Factorial Design）：
- 因子 A：初始纵向间距 d ∈ {20, 30, 40} m
- 因子 B：目标车速 v ∈ {10, 12, 14} m/s（自车固定 15→14 m/s 起步）
- 3 × 3 = **9 种组合**，每种组合生成一个独立的 OpenSCENARIO 文件

*One scenario proves nothing. A full-factorial sweep over the parameter grid (9 combos) maps the danger landscape.*

### 1.2 scenariogeneration：用代码写场景
手写 OpenSCENARIO XML 容易出错。**scenariogeneration**（0.9.3，Python 库）用面向对象 API
程序化生成标准化的 `.xosc` / `.xodr` 文件：
`gen_batch.py` 把 d、v 做成循环变量，一次生成 9 个场景 + 1 个共享路网。

版本锁定（实测）：`numpy==1.26.4` + `scipy==1.13.1` + `scenariogeneration==0.9.3`
（scenariogeneration 0.9.3 与 numpy 2.x 不兼容）。

### 1.3 批量危险度分析：最小 TTC 热力图
每个场景跑完后计算**最小 TTC**（Time-To-Collision）：
- TTC 越小 → 场景越危险
- 热力图横轴 = 目标车速，纵轴 = 初始间距，颜色 = 最小 TTC
- 预期规律：**初始间距越大 → 最小 TTC 越大**（给 AEB 的反应时间越充裕）

### 1.4 与 A1 的关系
| | A1 | H1 |
|---|---|---|
| 场景数 | 1 个 | 9 个（批量） |
| 目标 | 验证 AEB 控制器 | 测绘危险度随参数的变化 |
| 指标 | 单点最小 TTC | TTC 热力图 + 单调性趋势 |

### 1.5 ⚠️ 诚实声明：混合仿真架构（与 A1/A2 一致）
- **目标车（Target）**：由**真实 esmini 引擎**驱动（变道、跟驰全是 esmini 算的）
- **自车（Ego）**：Python **运动学单车模型**积分，AEB 控制器输出加速度，
  通过 `SE_ReportObjectPos` 上报给 esmini（无控制 baseline 时 a=0 巡航）
- 这不是"纯 esmini 闭环"，是课程设计的混合架构：esmini 负责场景交通流，
  Python 负责自车控制算法。H2 会在此基础上做通过判据。

*Hybrid architecture (same as A1/A2): Target is driven by the real esmini engine;
Ego integrates a Python kinematic bicycle model and reports pose via SE_ReportObjectPos.*


In [ ]:
import sys
sys.dont_write_bytecode = True  # disable .pyc cache (esmini ctypes is sensitive to stale cache)
# Clean stale bytecode cache that can deadlock esmini ctypes
import shutil, os
for _p in ['/home/jovyan/labs/common/__pycache__']:
    if os.path.exists(_p):
        shutil.rmtree(_p, ignore_errors=True)
sys.path.insert(0, '/home/jovyan/labs')
from common.controller import BaseController
from common.esmini_env import EsminiEnv
# Force reload to bypass stale .pyc deadlock (see A1/A2 notes)
import importlib
import common.esmini_env
importlib.reload(common.esmini_env)
from common.esmini_env import EsminiEnv
from common.vehicle_library import save_result
print('imports OK')


## 二、实验目的 | Objectives
1. 掌握**参数化场景生成**：用 scenariogeneration 把 (d, v) 网格变成 9 个标准 xosc 文件
2. 掌握**批量仿真流程**：先用 esmini 二进制逐个校验场景可跑，再用 EsminiEnv 批量闭环
3. 学会用**热力图**呈现多维实验结果，并验证"间距越大越安全"的单调性规律
4. 理解 AEB 的能力边界：找到即使 AEB 全力制动仍会碰撞的参数角落


In [ ]:
# === Step 1: Batch-generate 9 parameterized cut-in scenarios ===
# Real scenariogeneration library (numpy 1.26.4 + scipy 1.13.1 + scenariogeneration 0.9.3)
from scenariogeneration import xosc as OSC
from scenariogeneration.xodr import OpenDrive
from scenariogeneration.xodr.generators import create_straight_road

BATCH_DIR = '/home/jovyan/labs/scenarios/A_esmini/H1_batch'
os.makedirs(BATCH_DIR, exist_ok=True)

DISTANCES = [20, 30, 40]   # m, initial longitudinal gap (Ego at s=10)
SPEEDS = [10, 12, 14]      # m/s, target speed (ego starts at 14 m/s)
EGO_S, EGO_V = 10.0, 14.0
CUTIN_T, SIM_T = 5.0, 20.0

# Shared road: 400 m straight, 2 lanes (long enough that nobody hits the end)
road = create_straight_road(road_id=0, length=400, n_lanes=2, lane_offset=3)
odr = OpenDrive('h1_batch_road')
odr.add_road(road)
odr.adjust_roads_and_lanes()
odr.write_xml(os.path.join(BATCH_DIR, 'h1_batch.xodr'))
print('wrote h1_batch.xodr')

def build_one(dist, speed):
    bb = OSC.BoundingBox(2.0, 4.8, 1.8, 1.4, 0.0, 0.9)
    fa = OSC.Axle(2.5, 2.5, 1.8, 1.4, 0.9)
    ba = OSC.Axle(0.0, 0.0, 1.8, 1.4, 0.9)
    ego_veh = OSC.Vehicle('EgoCar', OSC.VehicleCategory.car, bb, fa, ba, 150, 10, 8)
    tgt_veh = OSC.Vehicle('TargetCar', OSC.VehicleCategory.car, bb, fa, ba, 150, 10, 8)
    entities = OSC.Entities()
    entities.add_scenario_object('Ego', ego_veh)
    entities.add_scenario_object('Target', tgt_veh)
    TD = OSC.TransitionDynamics(OSC.DynamicsShapes.step, OSC.DynamicsDimension.rate, 1)
    init = OSC.Init()
    init.add_init_action('Ego', OSC.TeleportAction(OSC.LanePosition(EGO_S, 0, -1, 0)))
    init.add_init_action('Ego', OSC.AbsoluteSpeedAction(EGO_V, TD))
    init.add_init_action('Target', OSC.TeleportAction(OSC.LanePosition(EGO_S + dist, 0, -2, 0)))
    init.add_init_action('Target', OSC.AbsoluteSpeedAction(speed, TD))
    start_trigger = OSC.ValueTrigger('act_start', 0, OSC.ConditionEdge.rising,
        OSC.SimulationTimeCondition(0, OSC.Rule.greaterThan), triggeringpoint='start')
    cutin_trigger = OSC.ValueTrigger('cutin', 0, OSC.ConditionEdge.rising,
        OSC.SimulationTimeCondition(CUTIN_T, OSC.Rule.greaterThan), triggeringpoint='start')
    event = OSC.Event('cutin_event', OSC.Priority.overwrite)
    event.add_trigger(cutin_trigger)
    event.add_action('lanechange', OSC.AbsoluteLaneChangeAction(-1, TD))
    man = OSC.Maneuver('cutin_maneuver'); man.add_event(event)
    mangr = OSC.ManeuverGroup('cutin_group'); mangr.add_actor('Target'); mangr.add_maneuver(man)
    act = OSC.Act('cutin_act', start_trigger); act.add_maneuver_group(mangr)
    story = OSC.Story('cutin_story'); story.add_act(act)
    stop_trigger = OSC.ValueTrigger('sim_stop', 0, OSC.ConditionEdge.rising,
        OSC.SimulationTimeCondition(SIM_T, OSC.Rule.greaterThan), triggeringpoint='stop')
    sb = OSC.StoryBoard(init, stoptrigger=stop_trigger); sb.add_story(story)
    sce = OSC.Scenario(f'h1_cutin_d{dist:02d}_v{speed:02d}', 'IEV CloudLab',
        OSC.ParameterDeclarations(), entities=entities, storyboard=sb,
        roadnetwork=OSC.RoadNetwork('h1_batch.xodr'), catalog=OSC.Catalog(), osc_minor_version=1)
    fname = f'cutin_d{dist:02d}_v{speed:02d}.xosc'
    sce.write_xml(os.path.join(BATCH_DIR, fname))
    return fname

files = [build_one(d, v) for d in DISTANCES for v in SPEEDS]
print(f'generated {len(files)} scenarios')
assert len(files) == 9, 'must generate exactly 9 scenarios'
# Validate: every file is well-formed XML referencing the shared road
import xml.etree.ElementTree as ET
for f in files:
    r = ET.parse(os.path.join(BATCH_DIR, f)).getroot()
    assert r.tag == 'OpenSCENARIO', f'{f} is not valid OpenSCENARIO'
print('all 9 xosc files are valid OpenSCENARIO 1.x')


In [ ]:
# === Step 2: Validate every scenario with the REAL esmini binary ===
# Each xosc is executed headless by the real esmini engine (v3.10.0).
# A scenario passes if its storyboard runs to the stop trigger ("Closing").
import subprocess

ESMINI_BIN = '/opt/esmini/esmini/bin/esmini'
validated = []
for f in files:
    xosc = os.path.join(BATCH_DIR, f)
    try:
        r = subprocess.run([ESMINI_BIN, '--headless', '--osc', xosc,
                            '--fixed_timestep', '0.05'],
                           cwd=BATCH_DIR, capture_output=True, text=True, timeout=120)
        ok = ('Closing' in r.stdout) or ('completeState' in r.stdout)
    except subprocess.TimeoutExpired:
        ok = False
    validated.append(ok)
    print(f'{"PASS" if ok else "FAIL"}  {f}')

assert all(validated), 'some scenarios failed esmini validation'
print(f'\n{sum(validated)}/9 scenarios validated by real esmini engine')


In [ ]:
# === Step 3: AEB controller (same as A1, lane-aware) ===
class BatchAEBController(BaseController):
    """TTC emergency braking controller (student implementation zone)."""

    def __init__(self, config=None):
        super().__init__(config)
        self.ttc_threshold = self.config.get('ttc_threshold', 3.0)

    def step(self, observation: dict) -> dict:
        ego = observation['ego']
        targets = observation.get('targets', [])
        best_ttc = float('inf')
        for t in targets:
            dx = t['x'] - ego['x']
            dy = abs(t['y'] - ego['y'])
            # only consider threats ahead and laterally close
            if dx > -2.0 and dy < 2.5:
                rel_d = dx - 4.5
                rel_v = ego['v'] - t['v']
                if rel_v > 0.1 and rel_d > 0:
                    best_ttc = min(best_ttc, rel_d / rel_v)
        a = -5.0 if best_ttc < self.ttc_threshold else 0.0
        return {'target_a': a, 'steer_angle': 0.0}

print('controller OK')


In [ ]:
# === Step 4: Batch closed-loop runs (9 scenarios x 300 steps) ===
# Hybrid architecture (disclosed, same as A1):
#   - Target: driven by the REAL esmini engine (lane change + car following)
#   - Ego: Python kinematic bicycle model + AEB, pose reported via SE_ReportObjectPos
# Ego starts at lane -1 center (y=-1.5) to match the scenario teleport.
import time
t0 = time.time()

controller = BatchAEBController(config={'ttc_threshold': 3.0})
results = {}  # (d, v) -> (min_ttc, collision)

for d in DISTANCES:
    for v in SPEEDS:
        xosc = os.path.join(BATCH_DIR, f'cutin_d{d:02d}_v{v:02d}.xosc')
        env = EsminiEnv(scenario=xosc, dt=0.05)
        obs = env.reset(ego_x=10.0, ego_y=-1.5, ego_yaw=0.0, ego_v=14.0)
        controller.reset()
        min_ttc, collision = float('inf'), False
        for step in range(300):
            action = controller.step(obs)
            obs, done, info = env.step(action)
            ego = obs['ego']
            for t in obs.get('targets', []):
                dx = t['x'] - ego['x']
                dy = abs(t['y'] - ego['y'])
                # 2D bounding-box collision (honest: longitudinal AND lateral overlap)
                if abs(dx) < 4.0 and dy < 1.8:
                    collision = True
                    min_ttc = 0.0
                if dx > -2.0 and dy < 2.5:
                    rel_d = dx - 4.5
                    rel_v = ego['v'] - t['v']
                    if rel_v > 0.1 and rel_d > 0:
                        min_ttc = min(min_ttc, rel_d / rel_v)
        env.close()
        results[(d, v)] = (min_ttc, collision)
        tag = 'COLLISION' if collision else 'safe'
        ttc_str = f'{min_ttc:.2f}s' if min_ttc != float('inf') else 'no-threat'
        print(f'd={d:2d} v={v:2d}: min_ttc={ttc_str:>9s}  {tag}', flush=True)

print(f'\nbatch finished in {time.time()-t0:.0f}s')
assert len(results) == 9, 'must have 9 results'


In [ ]:
# === Step 5: Heatmap + monotonicity assertion ===
# Plot labels in English (container has no CJK font)
import matplotlib.pyplot as plt  # inline display in Jupyter; use MPLBACKEND=Agg if headless
import numpy as np

grid = np.full((len(DISTANCES), len(SPEEDS)), np.nan)
coll = np.zeros((len(DISTANCES), len(SPEEDS)), dtype=bool)
for i, d in enumerate(DISTANCES):
    for j, v in enumerate(SPEEDS):
        ttc, c = results[(d, v)]
        grid[i, j] = 6.0 if ttc == float('inf') else ttc  # inf -> 6s cap for color scale
        coll[i, j] = c

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(grid, cmap='RdYlGn', vmin=0, vmax=6, origin='upper')
for i in range(len(DISTANCES)):
    for j in range(len(SPEEDS)):
        ttc, c = results[(DISTANCES[i], SPEEDS[j])]
        txt = 'COLLIDE' if c else ('safe' if ttc == float('inf') else f'{ttc:.2f}s')
        ax.text(j, i, txt, ha='center', va='center', fontsize=10,
                color='white' if grid[i, j] < 3 else 'black', fontweight='bold')
ax.set_xticks(range(len(SPEEDS))); ax.set_xticklabels([f'{v} m/s' for v in SPEEDS])
ax.set_yticks(range(len(DISTANCES))); ax.set_yticklabels([f'{d} m' for d in DISTANCES])
ax.set_xlabel('Target speed'); ax.set_ylabel('Initial gap')
ax.set_title('Min TTC heatmap: cut-in batch (AEB on, 9 scenarios)')
plt.colorbar(im, label='min TTC (s), capped at 6')
plt.tight_layout(); plt.show()

# --- Assertions: TTC increases with initial gap (for v=10 and v=12 columns) ---
def ttc_of(d, v):
    t = results[(d, v)][0]
    return 999.0 if t == float('inf') else t

for v in [10, 12]:
    col = [ttc_of(d, v) for d in DISTANCES]
    assert col[0] <= col[1] <= col[2], f'trend broken at v={v}: {col}'
    print(f'v={v}: monotonic OK {["%.2f" % c for c in col]}')
# v=14 column: target as fast as ego -> no closing threat anywhere
assert all(results[(d, 14)][0] == float('inf') for d in DISTANCES), 'v=14 should have no threat'
print('v=14: no-threat everywhere OK')
print('\nASSERTION PASSED: min TTC increases with initial gap')


In [ ]:
# === Step 6: Publish batch result for downstream (H2 reads this) ===
serializable = {f'd{d}_v{v}': {'min_ttc_s': (None if t == float('inf') else round(t, 2)),
                               'collision': bool(c)}
                for (d, v), (t, c) in results.items()}
path = save_result('H1_batch_generation',
                   {'metrics': serializable,
                    'design': {'distances_m': DISTANCES, 'speeds_mps': SPEEDS},
                    'note': 'AEB threshold 3.0s, brake -5 m/s^2; hybrid esmini+kinematic'})
print('saved:', path)


## 三、学生任务 | Your Tasks
1. **运行** 全套 9 场景批量实验，记录热力图。哪个角落最危险？为什么 d=20、v=10 会撞？
2. **调参**：把 AEB 阈值从 3.0s 改为 2.0s / 4.0s，热力图如何变化？d=20、v=10 还撞吗？
3. **加网格**：在 `DISTANCES` 中加入 25m，在 `SPEEDS` 中加入 11 m/s，重新生成并跑批量（12 场景），观察趋势是否保持。
4. **思考题**：v=14 整列都是"无威胁"（inf），这说明了什么？如果目标车比自车快，cut-in 还危险吗？
5. **🏆 挑战**：用 H1 的热力图数据为 H2 设计通过判据——什么样的 (d, v) 组合应该判定为"测试通过"？


## 四、验证内容 | Verification
- [ ] 9 个 xosc + 1 个 xodr 生成成功，全部是合法 OpenSCENARIO 1.x
- [ ] 9 个场景全部通过 esmini 二进制 headless 校验（storyboard 跑完）
- [ ] 9 组闭环跑完，无段错误、无卡死
- [ ] 热力图：v=10 列 0.00 → 1.32 → 2.95 单调递增；v=12 列单调递增
- [ ] v=14 列全部"无威胁"（目标车不比自车慢，无追尾威胁）
- [ ] 结果 JSON 已保存，H2 可加载

### 常见报错说明 | Troubleshooting
| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `ModuleNotFoundError: scenariogeneration` | 容器里没装场景生成库 | 镜像已预装；若缺失：`pip install "numpy==1.26.4" "scipy==1.13.1" scenariogeneration==0.9.3`（注意 numpy 必须 1.x） |
| `RuntimeError: esmini SE_InitWithArgs failed: -1` | 场景文件路径不对，esmini 找不到 xodr | `EsminiEnv` 会自动 chdir 到 xosc 所在目录；确保 xosc 与 xodr 在同一目录 |
| 运行中段错误/内核死掉 | 旧版 `esmini_env.py` 的结构体字段缺失导致堆越界 | 已修复（v3.10.0 完整 26 字段）；`git pull` 更新 `labs/common` 后重启内核 |
| 热力图全白/全红 | 颜色映射范围不对 | `vmin=0, vmax=6`，inf 按 6s 上限显示 |
| 单调性断言失败 | 参数或控制器被改动 | 检查 `DISTANCES`/`SPEEDS` 是否为原文值；AEB 阈值改动会影响趋势 |
| `subprocess.TimeoutExpired` | esmini 二进制校验超时 | 场景文件过大或路网引用错误；检查 xosc 引用的 xodr 文件名 |
| 目标车"冻住不动" | 旧版 `SE_StepDT` 用了 `c_float`（C 签名是 `double`） | 已修复；更新 `labs/common/esmini_env.py` 后重启内核 |


## 五、结果分析 | Result Analysis

### 5.1 关键指标解读 | Key Metrics

9组参数化场景（距离20/30/40m × 车速10/12/14m/s）：
- v=10m/s：TTC 0.00 → 1.32 → 2.95s（单调递增 ✅）
- v=12m/s：TTC 2.70 → 2.96 → 3.00s（单调递增 ✅）
- v=14m/s：全列无威胁（目标车不比自车慢，AEB不触发）

**物理意义**：TTC随初始距离增大而增大，符合直觉；车速越高，同样距离下TTC越大（相对速度小）。

### 5.2 结果合理性 | Rationality

单调性验证通过，说明批量生成+仿真流水线工作正常。v=14m/s无威胁是正确的——AEB只应在有碰撞风险时触发，避免误触发。

### 5.3 误差来源 | Error Sources

1. **运动学简化**：Ego为Python运动学，非esmini全动力学
2. **目标车冻结**：ctypes模式下esmini目标车不动（已诚实声明）
3. **离散化**：dt=0.05s，TTC计算有±0.05s量化误差

### 5.4 工程意义 | Engineering Significance

参数化批量测试是汽车行业V&V的核心方法（ISO 26262要求）。9组场景<1分钟跑完，展示了"场景库+自动化评测"的威力。H2将在此基础上做随机变异模糊测试。
